# RAG avec LangChain, FAISS et un modèle Hugging Face local

## Objectif
Construire un système de question-réponse RAG (Retrieval-Augmented Generation) simple, avec LangChain, un dataset public comme base de connaissances, et un petit modèle Hugging Face qui tourne en local — pas de clé API nécessaire.

## Ce que tu vas apprendre
Ce qu'est un vector store, pourquoi un retriever est essentiel dans un RAG, pourquoi il faut découper ("chunker") les documents avant de les embedder, comment les paramètres de chunking changent ce qui est récupéré, comment assembler une chaîne `RetrievalQA` avec LangChain, et comment déboguer un système RAG en inspectant les chunks récupérés et leurs sources.

---

**Je vais jouer mon rôle de mentor sans complaisance dès l'intro, parce que l'énoncé passe sous silence des points qui vont te faire perdre du temps si tu ne les anticipes pas :**

1. **`RetrievalQA` est une API dépréciée dans les versions récentes de LangChain** (remplacée par `create_retrieval_chain` + LCEL). Elle fonctionne encore, mais si tu vois un `DeprecationWarning`, ce n'est pas un bug de ta part — c'est LangChain qui te pousse vers une autre API. Je garde `RetrievalQA` ici parce que c'est ce que l'énoncé demande explicitement, mais sache que ce n'est plus la voie recommandée à long terme.
2. **`google/flan-t5-small` est un modèle minuscule (80M de paramètres) avec une fenêtre de contexte courte (~512 tokens).** Si tu passes `k=4` ou `k=6` chunks au chain_type par défaut (`"stuff"`, qui concatène tout le contexte dans un seul prompt), tu vas très probablement dépasser cette limite. LangChain ne te préviendra pas forcément bruyamment — le tokenizer va tronquer silencieusement, et le modèle risque d'ignorer une partie du contexte que tu penses lui avoir donné. Ne confonds pas "le modèle donne une mauvaise réponse" avec "le modèle n'a pas halluciné" : ici, il se peut qu'il n'ait tout simplement pas *vu* l'info.
3. **`flan-t5-small` est un modèle seq2seq, pas causal.** Le pipeline Hugging Face à utiliser est `"text2text-generation"`, pas `"text-generation"`. Erreur facile à faire si tu généralises depuis les exercices précédents avec GPT-2.
4. **`m-ric/huggingface_doc` est un dataset de documentation technique Hugging Face.** Poser des questions dessus qui n'ont clairement rien à voir avec cette doc (ex. "quelle est la capitale de la France ?") ne te dira rien d'utile sur ton système RAG — la question test doit être choisie pour être *réellement présente* dans les 200 premiers documents, sinon tu testes le comportement du modèle en absence de contexte pertinent, pas la qualité de ta récupération.

Je te préviens aussi : ce notebook n'est PAS validé par exécution réelle de ma part (pas d'accès réseau ici pour télécharger `m-ric/huggingface_doc` ni les modèles Hugging Face). Le code suit l'API documentée de LangChain/Transformers au moment de la rédaction, mais si une bibliothèque a changé de signature entre-temps, tu peux tomber sur une erreur que je n'ai pas anticipée. Ne prends pas ce notebook comme une vérité absolue — débogue-le comme n'importe quel code que tu n'as pas écrit toi-même.

## 1. Installation et imports

In [ ]:
%pip install -q datasets transformers sentence-transformers faiss-cpu
%pip install -q langchain langchain-community langchain-huggingface

In [ ]:
from datasets import load_dataset
from transformers import pipeline

from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA

# Ces deux imports viennent de packages séparés depuis les refactors récents de LangChain.
# Si l'un des deux échoue chez toi (version différente installée), essaie l'import
# depuis langchain_community à la place.
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS

## 2. Charger le dataset

In [ ]:
raw_dataset = load_dataset("m-ric/huggingface_doc", split="train[:200]")

print("Colonnes disponibles :", raw_dataset.column_names)
print("\nExemple de ligne brute :")
print(raw_dataset[0])

**Vérifie toi-même** avant de continuer : est-ce que la colonne `text` contient vraiment du texte exploitable (pas de valeurs vides, pas de HTML brut illisible) ? Ne suppose pas que parce que ça vient de Hugging Face, c'est forcément propre.

## 3. Convertir les lignes en objets `Document` LangChain

In [ ]:
documents = [
    Document(
        page_content=row["text"],
        metadata={"source": row["source"]}
    )
    for row in raw_dataset
    if row["text"]  # on écarte les lignes vides, l'énoncé ne le précise pas mais c'est une hygiène de base
]

print(f"{len(documents)} documents LangChain créés (sur {len(raw_dataset)} lignes brutes).")
print(documents[0].metadata)

## 4. Découper les documents en chunks

On teste **deux configurations différentes** volontairement, comme demandé, pour observer concrètement l'effet du chunking sur ce qui est récupéré plus tard.

In [ ]:
# Configuration A : chunks courts, peu de chevauchement
splitter_a = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks_a = splitter_a.split_documents(documents)

# Configuration B : chunks plus longs, chevauchement plus généreux
splitter_b = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks_b = splitter_b.split_documents(documents)

print(f"Config A (200/20)  : {len(chunks_a)} chunks, longueur moyenne = {sum(len(c.page_content) for c in chunks_a)/len(chunks_a):.0f} caractères")
print(f"Config B (800/100) : {len(chunks_b)} chunks, longueur moyenne = {sum(len(c.page_content) for c in chunks_b)/len(chunks_b):.0f} caractères")

**Point que tu dois comprendre, pas juste observer** : des chunks courts (config A) donnent une récupération plus précise (chaque chunk parle d'une seule chose) mais risquent de couper une explication en plein milieu. Des chunks longs (config B) gardent le contexte intact mais diluent la similarité — un chunk qui parle de 3 sujets différents matche moins bien une requête très spécifique. Il n'y a pas de "bonne" config universelle, seulement un compromis à choisir selon ton cas d'usage. Si tu ne peux pas expliquer ce compromis avec tes mots après cette cellule, relis la doc de `RecursiveCharacterTextSplitter` avant de continuer.

On continue avec la **configuration B** pour la suite du notebook (contexte plus complet, plus adapté à un modèle de génération), mais garde `chunks_a` sous la main pour comparer plus tard si tu veux.

In [ ]:
chunks = chunks_b

## 5. Construire le vector store et le retriever

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = FAISS.from_documents(chunks, embedding_model)

In [ ]:
retriever_k4 = vector_store.as_retriever(search_kwargs={"k": 4})
retriever_k2 = vector_store.as_retriever(search_kwargs={"k": 2})
retriever_k6 = vector_store.as_retriever(search_kwargs={"k": 6})

In [ ]:
test_question = "How do I use the Trainer class to fine-tune a model?"

for label, retriever in [("k=2", retriever_k2), ("k=4", retriever_k4), ("k=6", retriever_k6)]:
    retrieved = retriever.invoke(test_question)
    print(f"--- {label} : {len(retrieved)} chunks récupérés ---")
    for doc in retrieved:
        print(f"  source={doc.metadata.get('source')} | longueur={len(doc.page_content)}")
    print()

**Ce que tu dois vérifier, pas juste lire** : est-ce que les chunks supplémentaires quand `k` augmente sont réellement pertinents, ou sont-ils juste "les moins mauvais parmi ce qui reste" ? FAISS te renverra toujours *k* résultats, même s'il n'y a que 2 documents vraiment pertinents dans tout le corpus. Un `k` trop grand ne fait pas qu'ajouter du contexte utile — il peut aussi diluer le signal avec du bruit, surtout que `flan-t5-small` a une fenêtre de contexte courte (rappel du point 2 de l'intro).

## 6. Vérification de la récupération (obligatoire avant de générer quoi que ce soit)

In [ ]:
retriever = retriever_k4

sanity_question = "What is a tokenizer used for?"
retrieved_docs = retriever.invoke(sanity_question)

print(f"Question : {sanity_question}\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} (source: {doc.metadata.get('source')}) ---")
    print(doc.page_content[:400])
    print()

**Ne saute pas cette étape, même si elle te paraît redondante avec la cellule précédente.** Si les chunks affichés ici ne contiennent visiblement pas la réponse à la question posée, la génération qui suit ne pourra pas être bonne — et si elle l'est quand même "par chance" via les connaissances internes du modèle, tu n'as PAS construit un RAG fonctionnel, tu as juste un modèle qui répond avec ce qu'il savait déjà avant même de voir tes documents. C'est exactement le genre de confusion que l'exercice veut t'apprendre à détecter. Si les chunks ne sont pas pertinents ici, reviens en arrière et ajuste `chunk_size`/`chunk_overlap`/`k` avant de continuer — ne te contente pas de continuer en espérant que ça s'arrange.

## 7. Question-réponse RAG avec un modèle local

In [ ]:
# flan-t5-small est un modèle seq2seq (encodeur-décodeur), pas un modèle causal.
# Le pipeline correspondant est "text2text-generation", pas "text-generation".
hf_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    max_new_tokens=256
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True
)

In [ ]:
questions = [
    "What is a tokenizer used for?",
    "How do I use the Trainer class to fine-tune a model?",
    "What is the difference between a fast tokenizer and a slow tokenizer?"
]

for question in questions:
    result = qa_chain.invoke({"query": question})

    print(f"Question : {question}")
    print(f"Réponse  : {result['result']}")
    print("Sources utilisées :")
    for doc in result["source_documents"]:
        print(f"  - {doc.metadata.get('source')}")
    print("-" * 60)

## Bilan sans complaisance

- Si les réponses de `flan-t5-small` te paraissent courtes, vagues ou à côté de la question, ne conclus pas trop vite "mon RAG ne marche pas". Regarde d'abord si le problème vient de la récupération (chunks affichés à l'étape 6 pas pertinents) ou de la génération (chunks pertinents, mais le petit modèle ne sait pas bien les synthétiser). Ce sont deux pannes différentes avec deux corrections différentes — mélanger les deux, c'est le réflexe de débutant à éviter.
- `flan-t5-small` (80M paramètres) est probablement le facteur limitant principal de ce pipeline, pas ta stratégie de chunking ni ton choix de `k`. Si tu veux juger la qualité de ta partie *retrieval* indépendamment de la faiblesse du modèle de génération, relis directement les chunks récupérés (étape 6) plutôt que la réponse générée — c'est plus fiable pour évaluer si ton RAG "trouve" la bonne info, même s'il ne la "formule" pas bien.
- Ce notebook ne mesure rien objectivement (pas de métrique de qualité de réponse, pas de comparaison avec/sans RAG). Tu juges à l'œil. C'est suffisant pour un exercice pédagogique, mais si tu veux un jour comparer deux configurations de chunking de façon rigoureuse, il te faudra un jeu de questions-réponses de référence et une métrique (ex. exact match, F1, ou un juge LLM), pas juste "ça a l'air mieux".